In [ ]:
# lakehouse and table to incrementally load
table_path = "DE_LH_BRONZE_ELC.Item_Ledger_Entry"

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

# Define Metadata Schema
metadata_schema = StructType([
    StructField("table_name", StringType(), False),    # Name of the table
    StructField("last_refresh", TimestampType(), True),# Last refresh timestamp
    StructField("source_query", StringType(), True),   # Query to successfully pull data into delta table
    StructField("created_at", TimestampType(), False), # Metadata entry creation timestamp
])

# Example Data for Metadata Table
metadata_data = [
    (
        # Table name
        "DE_LH_BRONZE_ELC.Item_Ledger_Entry", 

        # Last refresh
        datetime(2024, 11, 22, 12, 0, 0),

        # Source query
        """
        SELECT 
            [timestamp] AS timestamp,
            [Entry No_] AS Entry_No_,
            [Customer No_] AS Customer_No_,
            [Posting Date] AS Posting_Date,
            [Document Type] AS Document_Type,
            [Document No_] AS Document_No_,
            [Description] AS Description,
            [Currency Code] AS Currency_Code,
            [Sales (LCY)] AS Sales_LCY,
            [Profit (LCY)] AS Profit_LCY,
            [Inv_ Discount (LCY)] AS Inv_Discount_LCY,
            [Sell-to Customer No_] AS Sell_to_Customer_No_,
            [Customer Posting Group] AS Customer_Posting_Group,
            [Global Dimension 1 Code] AS Global_Dimension_1_Code,
            [Global Dimension 2 Code] AS Global_Dimension_2_Code,
            [Salesperson Code] AS Salesperson_Code,
            [User ID] AS User_ID,
            [Source Code] AS Source_Code,
            [On Hold] AS On_Hold,
            [Applies-to Doc_ Type] AS Applies_to_Doc_Type,
            [Applies-to Doc_ No_] AS Applies_to_Doc_No_,
            [Open] AS [Open_Status], -- Renamed to avoid keyword conflict
            [Due Date] AS Due_Date,
            [Pmt_ Discount Date] AS Pmt_Discount_Date,
            [Original Pmt_ Disc_ Possible] AS Original_Pmt_Disc_Possible,
            [Pmt_ Disc_ Given (LCY)] AS Pmt_Disc_Given_LCY,
            [Positive] AS Positive,
            [Closed by Entry No_] AS Closed_by_Entry_No_,
            [Closed at Date] AS Closed_at_Date,
            [Closed by Amount] AS Closed_by_Amount,
            [Applies-to ID] AS Applies_to_ID,
            [Journal Batch Name] AS Journal_Batch_Name,
            [Reason Code] AS Reason_Code,
            [Bal_ Account Type] AS Bal_Account_Type,
            [Bal_ Account No_] AS Bal_Account_No_,
            [Transaction No_] AS Transaction_No_,
            [Closed by Amount (LCY)] AS Closed_by_Amount_LCY,
            [Document Date] AS Document_Date,
            [External Document No_] AS External_Document_No_,
            [Calculate Interest] AS Calculate_Interest,
            [Closing Interest Calculated] AS Closing_Interest_Calculated,
            [No_ Series] AS No_Series,
            [Closed by Currency Code] AS Closed_by_Currency_Code,
            [Closed by Currency Amount] AS Closed_by_Currency_Amount,
            [Adjusted Currency Factor] AS Adjusted_Currency_Factor,
            [Original Currency Factor] AS Original_Currency_Factor,
            [Remaining Pmt_ Disc_ Possible] AS Remaining_Pmt_Disc_Possible,
            [Pmt_ Disc_ Tolerance Date] AS Pmt_Disc_Tolerance_Date,
            [Max_ Payment Tolerance] AS Max_Payment_Tolerance,
            [Last Issued Reminder Level] AS Last_Issued_Reminder_Level,
            [Accepted Payment Tolerance] AS Accepted_Payment_Tolerance,
            [Accepted Pmt_ Disc_ Tolerance] AS Accepted_Pmt_Disc_Tolerance,
            [Pmt_ Tolerance (LCY)] AS Pmt_Tolerance_LCY,
            [Amount to Apply] AS Amount_to_Apply,
            [IC Partner Code] AS IC_Partner_Code,
            [Applying Entry] AS Applying_Entry,
            [Reversed] AS Reversed,
            [Reversed by Entry No_] AS Reversed_by_Entry_No_,
            [Reversed Entry No_] AS Reversed_Entry_No_,
            [Complaint No_] AS Complaint_No_,
            [Credit Memo No_] AS Credit_Memo_No_,
            [Credit] AS Credit,
            [Credit Posted] AS Credit_Posted,
            [NOT USED Your Reference] AS NOT_USED_Your_Reference,
            [Dispute] AS Dispute,
            [Dispute Date] AS Dispute_Date,
            [ELC Doc_ Type] AS ELC_Doc_Type,
            [Responsibility Center] AS Responsibility_Center,
            [Floor Plan] AS Floor_Plan,
            [Checked] AS Checked,
            [Journal Template Name] AS Journal_Template_Name,
            [Transaction Mode Code] AS Transaction_Mode_Code,
            [Bank Account Code] AS Bank_Account_Code,
            [Promised Payment] AS Promised_Payment,
            [Dispute Code] AS Dispute_Code,
            [Credit Type] AS Credit_Type,
            [Financing No_] AS Financing_No_,
            [ELC Document No_] AS ELC_Document_No_,
            [Commission code] AS Commission_Code,
            [SO Document No_] AS SO_Document_No_,
            [Prepayment] AS Prepayment,
            [Redistributed] AS Redistributed
        FROM [copyofproduction].[dbo].[Production$Cust_ Ledger Entry];
        """,

        # Date created
        datetime.now()
    ),
]

# Create DataFrame
metadata_df = spark.createDataFrame(metadata_data, metadata_schema)

# Save the DataFrame as a Delta Table in the Tables Section
table_name = "table_refresh_metadata"  # Name of the table in the Lakehouse

metadata_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"Metadata table created in the Tables section with name: {table_name}")

StatementMeta(, 6e96690a-4a3e-461b-b14f-cb4b0d1bca41, 3, Finished, Available, Finished)

Metadata table created in the Tables section with name: table_refresh_metadata


In [3]:
# Load the Metadata Table
metadata_table = spark.table("table_refresh_metadata")

# Display the Data
metadata_table.show()



StatementMeta(, 02798105-8d6d-4fcc-9912-636a6f93b268, 5, Finished, Available, Finished)

+-------------+-------------------+---------+--------------------+--------------------+
|   table_name|       last_refresh|row_count|              schema|          created_at|
+-------------+-------------------+---------+--------------------+--------------------+
|example_table|2024-11-22 12:00:00|      100|{"column1":"strin...|2024-11-25 13:48:...|
+-------------+-------------------+---------+--------------------+--------------------+



# Get the last update date

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import LongType
import struct

# Initialize Spark session with the necessary configuration
spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
    .getOrCreate()

# Load data into the dataframe
df = spark.read.table("DE_LH_BRONZE_ELC.Item_Ledger_Entry")

# Define a Pandas UDF to convert binary to bigint
@pandas_udf(LongType())
def binary_to_bigint_udf(binary_series):
    return binary_series.apply(lambda x: struct.unpack('>q', x)[0])

# Apply the Pandas UDF to create a new column with the bigint representation of the timestamp
df = df.withColumn("timestamp_bigint", binary_to_bigint_udf(col("timestamp")))

# Find the row with the maximum 'timestamp_bigint' and get the corresponding 'Entry_No_'
max_row = df.orderBy(col("timestamp_bigint").desc()).select("timestamp_bigint", "Entry_No_").first()
max_timestamp_sequence = max_row['timestamp_bigint']
max_timestamp_no = max_row['Entry_No_']

# Print the maximum timestamp sequence number and corresponding 'Entry_No_' value
print(f"Max timestamp sequence number: {max_timestamp_sequence}")
print(f"Corresponding [Entry_No_] value: {max_timestamp_no}")

# Set the whereClause parameter
mssparkutils.notebook.exit(f"{max_timestamp_sequence}")


StatementMeta(, 864afe77-7984-4ecc-bbfe-5398c2565274, 9, Finished, Available, Finished)

Max timestamp sequence number: 15871262650
Corresponding [Entry_No_] value: 17685131
ExitValue: 15871262650